<a href="https://colab.research.google.com/github/BioML-UGent/Advanced-AI-for-Bioinformatics/blob/main/01_Intro_Neural_Networks/01_intro_to_NNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PC lab 1: Introduction to neural networks & PyTorch

In the lecture, we built neural networks up from linear and logistic regression. In this PC lab, we translate those equations into code: first by hand with tensors, then with the building blocks of PyTorch. We end with a digit classifier that is still untrained. Training it is the topic of PC lab 2.

**Learning goals**
- Work with PyTorch tensors: shapes, data types, matrix multiplication and broadcasting
- Implement a neuron and a two-layer network directly from the equations of the lecture
- Build networks with `torch.nn` and write your own `nn.Module`
- Load data with `Dataset` and `DataLoader`
- Interpret the outputs of a network: logits, softmax probabilities and the cross-entropy loss


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1 Recap: from linear models to neural networks

A single neuron computes a weighted sum of its $D$ inputs plus a bias. This is called the **activation** $a_j$:

$$a_j = \sum_{i=1}^{D} w_{ji} x_i + w_{j0}$$

As in the lecture, we can absorb the bias into the weights by adding a dummy input $x_0 = 1$:

$$a_j = \sum_{i=0}^{D} w_{ji} x_i$$

The activation is then passed through a **nonlinear activation function** $h(\cdot)$, giving $z_j = h(a_j)$. Classic choices are the sigmoid $\sigma$ and $\tanh$. Nowadays, the default for hidden layers is the **ReLU**, $h(a) = \max(0, a)$, which is cheap to compute and has a gradient of 1 for all positive inputs.

A two-layer network with $M$ hidden units and $K$ outputs chains two of these transformations:

$$y_k(\mathbf{x}, \mathbf{w}) = f\left(\sum_{j=0}^{M} w^{(2)}_{kj}\, h\left(\sum_{i=0}^{D} w^{(1)}_{ji} x_i\right)\right)$$

or, in matrix notation for layer $l$: $\mathbf{z}^{(l)} = h^{(l)}\left(\mathbf{W}^{(l)} \mathbf{z}^{(l-1)}\right)$.

The output activation $f$ depends on the task:

| Task | Output activation $f$ | Output |
|---|---|---|
| Regression | none (identity) | $\mathbb{R}$ |
| Binary classification | sigmoid | one probability in $[0, 1]$ |
| Multi-class classification | softmax | $K$ probabilities that sum to 1 |
| Multi-label classification | sigmoid on every output | $K$ probabilities, each in $[0, 1]$ |

The figure below builds up a neural network step by step: linear regression (**a**), logistic regression (**b**), a single ReLU neuron (**c**), multi-output regression (**d**), multi-label classification (**e**), multi-class classification with a softmax (**f**), and finally a network with two hidden layers for binary classification (**g**).

<img src='https://raw.githubusercontent.com/BioML-UGent/MLLS/main/11_intro_nns/lr2nn.png'>

**The simplest neural network is a stack of linear regressions with nonlinearities in between.** Because every neuron is connected to every neuron in the next layer, this type of network is called a fully-connected network or **multi-layer perceptron (MLP)**.

If we draw the biases explicitly, an MLP looks like this:

<img src='https://raw.githubusercontent.com/BioML-UGent/MLLS/main/11_intro_nns/biases.png'>

<div class="alert alert-success">

<b>EXERCISE 0 (THINK)</b>

How many parameters (weights and biases) does the network in the figure above have? You will check your answer in code in Exercise 6.

</div>

**Your answer:**

## 2 PyTorch tensors

Several Python libraries make it easy to build neural networks (PyTorch, TensorFlow/Keras, JAX, ...). They provide ready-made layers, loss functions and, most importantly, automatic computation of gradients. In this course, we use [PyTorch](https://pytorch.org), the most popular deep learning library in research.

The core data structure of PyTorch is the **tensor**. Tensors behave like NumPy arrays, and almost every NumPy function has a PyTorch counterpart. The differences are that tensors can live on a GPU, which makes matrix multiplications orders of magnitude faster, and that PyTorch can track the operations on tensors to compute gradients (PC lab 2).

On Colab, PyTorch is pre-installed. To run this notebook locally, see the [installation instructions](https://pytorch.org/get-started/locally/).

### 2.1 Creating tensors

In [ ]:
x = [[5, 8], [9, 8]]
print(torch.tensor(x))
print(np.array(x))

# converting between NumPy and PyTorch
x_numpy = np.array(x)
x_torch = torch.from_numpy(x_numpy)
print(x_torch.numpy())

In [ ]:
print(torch.zeros(8, 50).shape)  # also: torch.ones
print(torch.rand(8, 50).shape)   # uniform on [0, 1)
print(torch.randn(8, 50).shape)  # standard normal distribution
print(torch.arange(6))

### 2.2 Data types

The default float type in PyTorch is `float32` (also called `torch.float`), while NumPy uses `float64` (`torch.double`). The weights of a PyTorch model are `float32`, so **convert NumPy data with `.float()`** before feeding it to a model. Class labels are usually stored as `int64` (`torch.long`).

In [ ]:
print(np.zeros(3).dtype, torch.zeros(3).dtype)

x = torch.from_numpy(np.zeros(3))
print(x.dtype, "->", x.float().dtype)

labels = torch.tensor([0, 2, 1])
print(labels.dtype)

### 2.3 Indexing, reshaping and reductions

Indexing and slicing work as in NumPy. `reshape` changes the shape of a tensor; a `-1` means "infer this dimension from the others".

In [ ]:
x = torch.randn(8, 50, 60)
print(x.shape)
print(x[:4, 10:-10].shape)
print(x[0, 0, :5])
print(x.reshape(8, -1).shape)

Reductions such as `sum`, `mean`, `std` and `max` take a `dim` argument: the dimension that is **collapsed**. For a data matrix of shape (samples, features), `dim=0` gives one value per feature.

In [ ]:
X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print(X.mean())       # over all elements
print(X.mean(dim=0))  # per column (feature): shape (3,)
print(X.mean(dim=1))  # per row (sample): shape (2,)

Tensors can be joined with `torch.cat` along an existing dimension:

In [ ]:
a = torch.zeros(2, 3)
b = torch.ones(2, 3)
print(torch.cat([a, b], dim=0).shape)  # add rows    -> (4, 3)
print(torch.cat([a, b], dim=1).shape)  # add columns -> (2, 6)

### 2.4 Matrix multiplication vs element-wise multiplication

`@` (or `torch.matmul`) is matrix multiplication; `*` is **element-wise** multiplication. For example, to linearly combine 8 samples with 26 features into one output per sample:

In [ ]:
X = torch.randn(8, 26)
w = torch.randn(26)
print((X @ w).shape)  # (8, 26) @ (26,) -> (8,)

A = torch.tensor([[1., 2.],
                  [3., 4.]])
print(A @ A)  # matrix product
print(A * A)  # element-wise product

### 2.5 Broadcasting

When the shapes of two tensors don't match, PyTorch (like NumPy) **broadcasts** them: missing leading dimensions and dimensions of size 1 are virtually repeated ([documentation](https://pytorch.org/docs/stable/notes/broadcasting.html)). A common use is subtracting the mean of every feature from a data matrix:

In [ ]:
X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])  # shape (2, 3)
col_means = X.mean(dim=0)         # shape (3,)
print(X - col_means)              # (2, 3) - (3,): the means are subtracted from every row

Whatever you want to do with a tensor, there is probably a function for it. Search the [documentation](https://pytorch.org/docs/stable/torch.html) before writing it yourself.

### Exercise 1: preparing a real dataset

We will use the [Wisconsin breast cancer dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset): 569 tumour samples, each described by 30 features of the cell nuclei (radius, texture, smoothness, ...) measured on digitized images of a fine-needle aspirate. The target is whether the tumour is malignant (0) or benign (1). The dataset ships with scikit-learn, so no download is needed.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_np, y_np = data.data, data.target
print(X_np.shape, y_np.shape)
print(data.feature_names[:5])
print(data.target_names)

<div class="alert alert-success">

<b>EXERCISE 1</b>

a) Convert `X_np` and `y_np` to tensors `X` and `y`. Use `float32` for the features and `long` for the labels.

b) Randomly split the data in a training set (80%) and a test set (20%). Call the results `X_train`, `y_train`, `X_test` and `y_test`. Hint: `torch.randperm(n)` returns a random permutation of `0, ..., n-1`.

c) Standardize the features (mean 0 and standard deviation 1 for every feature) using the mean and standard deviation of the <b>training set</b>, and overwrite `X_train` and `X_test` with the standardized features. Check that the training features now have mean ≈ 0 and std ≈ 1. What about the test features?

d) <b>THINK:</b> Why do we standardize the test set with the statistics of the training set, instead of its own statistics?

</div>

a)

In [ ]:
######## YOUR CODE HERE #########

#################################

b)

In [ ]:
######## YOUR CODE HERE #########

#################################

c)

In [ ]:
######## YOUR CODE HERE #########

#################################

d)

**Your answer:**

## 3 From the lecture's equations to code

Before using PyTorch's layers, we implement the equations of the lecture with plain tensor operations, using the standardized breast cancer data.

One detail about notation: we store data with **one sample per row**, as a matrix $\mathbf{X}$ of shape $(N, D)$. The lecture writes a single sample as a column vector, $\mathbf{a} = \mathbf{W}\mathbf{x}$, with $\mathbf{W}$ of shape $(M, D)$. For a whole dataset of row vectors, this becomes $\mathbf{A} = \mathbf{X}\mathbf{W}^\top$, of shape $(N, M)$. PyTorch uses the same convention, as we will see in section 4.

We do not train anything yet: all weights are random. Learning the weights is the topic of PC lab 2.

<div class="alert alert-success">

<b>EXERCISE 2: logistic regression is a single neuron</b>

Logistic regression is a single neuron with a sigmoid activation: $y = \sigma(\mathbf{w}^\top\mathbf{x} + w_0)$.

a) Draw random weights with `w = torch.randn(D) * 0.1` and `w0 = torch.zeros(1)`, where `D` is the number of features. Compute the predicted probability of "benign" for every training sample with `torch.sigmoid`. Check the shape and range of the output.

b) Classify a sample as benign when its probability is larger than 0.5, and compute the accuracy on the training set. Run the cell a few times, drawing new random weights each time. <b>THINK:</b> What do you notice?

</div>

In [ ]:
D = X_train.shape[1]  # number of features

######## YOUR CODE HERE #########

#################################

**Your answer:**

Below are random weights for a two-layer network with $D = 30$ inputs, $M = 16$ hidden units and $K = 2$ outputs (one for each class: malignant and benign). Note the shapes: (number of outputs, number of inputs), as in the lecture.

In [ ]:
M, K = 16, 2
W1 = torch.randn(M, D) * 0.1  # layer 1: (out, in)
b1 = torch.zeros(M)
W2 = torch.randn(K, M) * 0.1  # layer 2: (out, in)
b2 = torch.zeros(K)

<div class="alert alert-success">

<b>EXERCISE 3: a two-layer network by hand</b>

Compute the forward pass of the network with the weights above, using matrix multiplications and PyTorch's activation functions:

- the hidden units `Z` (shape $(N, M)$), with `torch.relu`,
- the output probabilities `Y` (shape $(N, K)$), with `torch.softmax`. Over which dimension should the softmax be computed?

Check that every row of `Y` sums to 1.

</div>

In [ ]:
######## YOUR CODE HERE #########

#################################

<div class="alert alert-success">

<b>EXERCISE 4 (THINK): what if we leave out the activation function?</b>

The lecture asked: *"How flexible is a deep neural network if the nonlinear activation function h is left out in every layer?"* What does the network of Exercise 3 reduce to without the ReLU? Answer first, then run the cell below the answer to check.

</div>

**Your answer:**

In [ ]:
# Without the ReLU, two linear layers can be merged into a single linear layer
out_two_layers = (X_train @ W1.T + b1) @ W2.T + b2

W = W2 @ W1       # (K, M) @ (M, D) -> (K, D)
b = W2 @ b1 + b2  # (K,)
out_one_layer = X_train @ W.T + b

print(torch.allclose(out_two_layers, out_one_layer, atol=1e-5))

## 4 Building networks with `torch.nn`

Implementing every layer by hand quickly becomes tedious. The [`torch.nn`](https://pytorch.org/docs/stable/nn.html) module provides ready-made building blocks. The most basic one is the [linear layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html), `nn.Linear(in_features, out_features)`. It computes `x @ W.T + b`, with a weight matrix of shape (out, in): exactly what we did in Exercise 3.

In [ ]:
layer = nn.Linear(30, 16)
print(layer)
print(layer.weight.shape, layer.bias.shape)

out = layer(X_train)
print(X_train.shape, "->", out.shape)

The number of input features of the data must match the layer. Otherwise, you get an error you will see often:

In [ ]:
try:
    layer(torch.randn(16, 20))
except RuntimeError as e:
    print("RuntimeError:", e)

`nn.Linear` initializes its weights randomly. We can overwrite them with our own values to check that it computes the same thing as our implementation.

<div class="alert alert-success">

<b>EXERCISE 5</b>

Create two `nn.Linear` layers matching the network of Exercise 3, and copy the weights `W1`, `b1`, `W2` and `b2` into them. Check that the layers give the same output probabilities as `Y` from Exercise 3.

Hint: copy weights with `layer.weight.copy_(W1)` inside a `with torch.no_grad():` block. This tells PyTorch not to track the operation for computing gradients (more on this in PC lab 2).

</div>

In [ ]:
######## YOUR CODE HERE #########

#################################

Activation functions are available as modules too, such as `nn.ReLU`, `nn.Sigmoid` and `nn.Tanh`:

In [ ]:
relu_layer = nn.ReLU()
x = torch.randn(2, 4)
print(x)
print(relu_layer(x))

`nn.Sequential` chains modules into a network. Always keep track of the dimensions: if a layer outputs 64 features, the next layer must take 64 inputs.

In [ ]:
model = nn.Sequential(
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
print(model)

x = torch.randn(8, 128)
z = model(x)
print(z.shape)
print(z)

The outputs of the last linear layer, before the output activation, are called **logits**. Notice that we did not put a sigmoid or softmax at the end of the network. In PyTorch, it is common for models to output logits and to let the loss function apply the output activation internally, because this is numerically more stable. When you need probabilities, apply `torch.sigmoid` or `torch.softmax` yourself.

(When you print the output of a model, you will see a `grad_fn`: PyTorch keeps track of the computations to compute gradients later. This is the topic of PC lab 2.)

<div class="alert alert-success">

<b>EXERCISE 6</b>

a) Build the network from the figure in section 1 (4 → 3 → 2 → 1, ReLU in the hidden layers, no output activation) with `nn.Sequential`.

b) Count its parameters with `sum(p.numel() for p in model.parameters())`, and compare with your answer to Exercise 0.

c) Print the name and shape of every parameter tensor, using `model.named_parameters()`.

</div>

In [ ]:
######## YOUR CODE HERE #########

#################################

## 5 Writing your own models

`nn.Sequential` works for simple chains of layers. Writing our own model class gives us more control: we can put any computation in the forward pass (like the residual connection below) and pass hyperparameters as arguments. A PyTorch model:
- is a subclass of `nn.Module`, and calls `super().__init__()` first,
- creates its layers in `__init__` as attributes, so that PyTorch registers their parameters,
- defines the computation in `forward(x)`, which runs when you call `model(x)`.

The code below shows two examples: a model with a fixed architecture, and a model where the layer sizes are a hyperparameter.

In [ ]:
class BasicModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(50, 40)
        self.layer2 = nn.Linear(40, 20)
        self.layer3 = nn.Linear(20, 5)
        self.relu = nn.ReLU()  # has no parameters, so we can reuse it

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.layer3(x)  # no activation after the last layer: the model outputs logits
        return x


class HyperparameterModel(nn.Module):
    def __init__(self, dimensions_from_input_to_output=(50, 40, 20, 10, 5), dropout=False):
        super().__init__()
        dims = dimensions_from_input_to_output

        layers = []
        # hidden layers: Linear -> ReLU (-> Dropout)
        for d_in, d_out in zip(dims[:-2], dims[1:-1]):
            layers.append(nn.Linear(d_in, d_out))
            layers.append(nn.ReLU())
            if dropout:
                layers.append(nn.Dropout(0.2))

        # output layer, without activation
        layers.append(nn.Linear(dims[-2], dims[-1]))

        # wrap the layers in a Sequential
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

The `dropout` option adds dropout layers, a regularization technique we will cover in PC lab 2. We leave it off for now.

In [ ]:
net = BasicModel()
print(net)
print(net(torch.randn(4, 50)).shape)

In [ ]:
net = HyperparameterModel(dimensions_from_input_to_output=[50, 160, 80, 40, 20, 10, 5])
print(net)
print(net(torch.randn(4, 50)).shape)

<div class="alert alert-success">

<b>EXERCISE 7</b>

Now it is time to create your own model. This model should contain a "residual block". <a href="https://en.wikipedia.org/wiki/Residual_neural_network">Residual connections</a> let the data skip some layers: the input of the block is added to the output of those layers. We will see why this is useful later in the course.

<img src='https://upload.wikimedia.org/wikipedia/commons/b/ba/ResBlock.png' width=400>

Create a model that takes 25 input features, has a residual block with two linear layers of 25 nodes each, and then reduces to the number of classes, which is given as a hyperparameter. Call your class `ResidualModel`, with the number of classes as argument `num_classes`.

</div>

In [ ]:
######## YOUR CODE HERE #########

#################################

In [ ]:
# Test your ResidualModel
model = ResidualModel(num_classes=5)
test_input = torch.randn(4, 25)
output = model(test_input)
print("Input shape: ", test_input.shape)
print("Output shape:", output.shape)  # should be (4, 5)

## 6 Data in PyTorch

### 6.1 `Dataset` and `DataLoader`

PyTorch's data loading revolves around two objects:
- a `Dataset` represents the whole dataset and returns one `(x, y)` sample at a time,
- a `DataLoader` iterates over a dataset in (optionally shuffled) mini-batches.

For data that is already stored in tensors, `TensorDataset` wraps them into a dataset. Let's try it on the breast cancer training data:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
print(len(train_dataset))
print(train_dataset[0])  # one (x, y) pair

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
for X_batch, y_batch in train_loader:
    print(X_batch.shape, y_batch.shape)

Notice that the last batch is smaller. To test code on a single batch, you can take the first batch with `next(iter(loader))`:

In [ ]:
X_batch, y_batch = next(iter(train_loader))
print(X_batch.shape, y_batch[:10])

We shuffle the training data so that every pass over the data (every epoch) sees the batches in a different order. For validation and test data, the order does not matter, so we use `shuffle=False`.

### 6.2 MNIST

[MNIST](https://en.wikipedia.org/wiki/MNIST_database) is a classic benchmark of 70,000 grayscale images of handwritten digits, each 28 × 28 pixels: the "raw pixels as features" example from the lecture. The `torchvision` package downloads it for us.

We store the MNIST data in separate variables (`X_train_mnist`, `y_train_mnist`, ...), to keep them apart from the breast cancer data.

In [ ]:
from torchvision import datasets

train_data = datasets.MNIST(root="data", train=True, download=True)
test_data = datasets.MNIST(root="data", train=False, download=True)

# the raw images and labels as tensors
X_train_mnist, y_train_mnist = train_data.data, train_data.targets
X_test_mnist, y_test_mnist = test_data.data, test_data.targets

print(X_train_mnist.shape, X_train_mnist.dtype, y_train_mnist.shape, y_train_mnist.dtype)
print(X_test_mnist.shape, y_test_mnist.shape)

The pixel values are integers between 0 and 255 (`uint8`). We convert them to floats between 0 and 1. The MLPs we have seen so far take a vector as input, so we also flatten every 28 × 28 image to a vector of 784 features. In PC lab 3, convolutional networks will use the 2D structure of the images instead.

In [ ]:
X_train_mnist = X_train_mnist.float() / 255
X_test_mnist = X_test_mnist.float() / 255

X_train_mnist = X_train_mnist.reshape(-1, 28 * 28)
X_test_mnist = X_test_mnist.reshape(-1, 28 * 28)

print(X_train_mnist.shape, X_train_mnist.dtype, X_train_mnist.min().item(), X_train_mnist.max().item())

We split off 20% of the training set as a validation set, and wrap every split in a data loader:

In [ ]:
perm = torch.randperm(len(X_train_mnist))
n_train = int(0.8 * len(X_train_mnist))
train_idx, val_idx = perm[:n_train], perm[n_train:]

X_val_mnist, y_val_mnist = X_train_mnist[val_idx], y_train_mnist[val_idx]
X_train_mnist, y_train_mnist = X_train_mnist[train_idx], y_train_mnist[train_idx]

train_loader = DataLoader(TensorDataset(X_train_mnist, y_train_mnist), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_mnist, y_val_mnist), batch_size=256, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_mnist, y_test_mnist), batch_size=256, shuffle=False)

print(len(X_train_mnist), len(X_val_mnist), len(X_test_mnist))

Let's visualize a batch:

In [ ]:
X_batch, y_batch = next(iter(train_loader))

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, img, label in zip(axes.flat, X_batch, y_batch):
    ax.imshow(img.reshape(28, 28), cmap="gray")
    ax.set_title(label.item())
    ax.axis("off")
plt.tight_layout()
plt.show()

<div class="alert alert-success">

<b>EXERCISE 8: an untrained digit classifier</b>

Let's put everything together.

a) Create a model that classifies MNIST digits. Make sure it has the right number of inputs and outputs; the rest is up to you.

b) Loop over the validation loader and collect the logits of all batches. Wrap the loop in `with torch.no_grad():`, since we don't need gradients. Concatenate the results with `torch.cat`.

c) Convert the logits to class probabilities with a softmax, and to predicted classes with `argmax`. Compute the validation accuracy.

d) In the lecture, we saw that fitting a classifier by maximum likelihood is equivalent to minimizing the <b>negative log-likelihood</b>. For multi-class classification, this is the <b>cross-entropy</b>:

$$\mathcal{L} = -\frac{1}{N}\sum_{n=1}^{N} \log p_{n, y_n}$$

where $p_{n, y_n}$ is the predicted probability of the true class of sample $n$. Compute it by hand, and compare with `nn.CrossEntropyLoss()(logits, y_val_mnist)`. Note that this loss function takes <b>logits</b> as input, not probabilities!

e) <b>THINK:</b> Which values do you get for the accuracy and the loss? Explain both numbers. (Hint: compute $\ln(10)$.)

</div>

a) and b)

In [ ]:
######## YOUR CODE HERE #########

#################################

c)

In [ ]:
######## YOUR CODE HERE #########

#################################

d)

In [ ]:
######## YOUR CODE HERE #########

#################################

e)

**Your answer:**